# Olist E-commerce — Basic Data Exploration and Data Quality Checks

This notebook uses beginner-friendly, reusable code to explore all nine CSV files and identify possible data-quality issues.

## Before running
Keep this notebook in the same folder as `Raw_Data`, which contains all nine CSV files. Run the cells from top to bottom.

**Important:** The checks flag possible issues; they do not automatically delete rows or fix values.

## 1. Import pandas

In [1]:
import pandas as pd

## 2. Load the nine CSV files

The file paths below use your `Raw_Data` folder.

In [2]:
customer_df = pd.read_csv(r"Raw_Data\olist_customers_dataset.csv")
geolocation_df = pd.read_csv(r"Raw_Data\olist_geolocation_dataset.csv")
order_items_df = pd.read_csv(r"Raw_Data\olist_order_items_dataset.csv")
order_payments_df = pd.read_csv(r"Raw_Data\olist_order_payments_dataset.csv")
order_reviews_df = pd.read_csv(r"Raw_Data\olist_order_reviews_dataset.csv")
orders_df = pd.read_csv(r"Raw_Data\olist_orders_dataset.csv")
products_df = pd.read_csv(r"Raw_Data\olist_products_dataset.csv")
sellers_df = pd.read_csv(r"Raw_Data\olist_sellers_dataset.csv")
product_category_df = pd.read_csv(r"Raw_Data\product_category_name_translation.csv")

# Keep all datasets in one dictionary so we can reuse the same code.
data = {
    "customers": customer_df,
    "geolocation": geolocation_df,
    "order_items": order_items_df,
    "order_payments": order_payments_df,
    "order_reviews": order_reviews_df,
    "orders": orders_df,
    "products": products_df,
    "sellers": sellers_df,
    "product_category": product_category_df
}

print("All nine CSV files loaded.")

All nine CSV files loaded.


## 3. Standardize column names

This converts column names to lowercase, removes spaces at the start/end, and replaces spaces between words with underscores.

In [3]:
for name, df in data.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )

print("Column names standardized.")

Column names standardized.


## 4. Quick overview of all datasets

Shows the number of rows and columns, duplicate rows, and missing cells.

In [4]:
overview = []

for name, df in data.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum()
    })

pd.DataFrame(overview)

,dataset,rows,columns,duplicate_rows,missing_cells
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,0,145903
5,orders,99441,8,0,4908
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,product_category,71,2,0,0


## 5. Check missing values and column details

This reusable function shows the first five rows, data types, non-missing values, missing values, unique-value counts, and numeric summary.

In [5]:
def explore_dataset(name, df):
    print("=" * 60)
    print("DATASET:", name)
    print("Rows and columns:", df.shape)

    print("\nFirst 5 rows:")
    display(df.head())

    print("\nColumn details:")
    details = pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "non_missing": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique()
    })
    display(details)

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nNumeric summary:")
    display(df.describe())


## 6. Run basic exploration on all nine datasets

In [6]:
for name, df in data.items():
    explore_dataset(name, df)

DATASET: customers
Rows and columns: (99441, 5)

First 5 rows:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Column details:


,data_type,non_missing,missing,missing_percent,unique_values
customer_id,str,99441,0,0.0,99441
customer_unique_id,str,99441,0,0.0,96096
customer_zip_code_prefix,int64,99441,0,0.0,14994
customer_city,str,99441,0,0.0,4119
customer_state,str,99441,0,0.0,27



Duplicate rows: 0

Numeric summary:


,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


... (44 outputs left out)


Column details:


,data_type,non_missing,missing,missing_percent,unique_values
product_category_name,str,71,0,0.0,71
product_category_name_english,str,71,0,0.0,71



Duplicate rows: 0

Numeric summary:


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


## 7. Print unique categorical values and check text inconsistencies

This checks text columns for extra spaces and possible differences in capitalization or spacing. It does not change the values.

In [7]:
def check_categorical_data(name, df):
    print("\nDATASET:", name)

    categorical_columns = df.select_dtypes(include="object").columns

    for col in categorical_columns:
        values = df[col].dropna().astype(str)

        print("\nColumn:", col)
        print("Unique values:", values.unique())

        # Count values with spaces at the beginning or end.
        extra_spaces = values.ne(values.str.strip()).sum()
        print("Values with extra spaces:", extra_spaces)

        # Group values after trimming spaces and ignoring capitalization.
        normalized = values.str.strip().str.lower()
        

## 8. Run categorical checks on all nine datasets

Some columns, such as IDs or city names, may contain many unique values, so this section can produce a lot of output.

In [8]:
for name, df in data.items():
    check_categorical_data(name, df)


DATASET: customers

Column: customer_id
Unique values: <StringArray>
['06b8999e2fba1a1fbc88172c00ba8bc7', '18955e83d337fd6b2def6b18a428ac77',
 '4e7b3e00288586ebd08712fdd0374a03', 'b2b6027bc5c5109e529d4dc6358b12c3',
 '4f2d8ab171c80ec8364f7c12e35b23ad', '879864dab9bc3047522c92c82e1212b8',
 'fd826e7cf63160e536e0908c76c3f441', '5e274e7a0c3809e14aba7ad5aae0d407',
 '5adf08e34b2e993982a47070956c5c65', '4b7139f34592b3a31687243a302fa75b',
 ...
 'be842c57a8c5a62e9585dd72f22b6338', 'f255d679c7c86c24ef4861320d5b7675',
 '14308d2303a3e2bdf4939b86c46d2679', 'f5a0b560f9e9427792a88bec97710212',
 '7fe2e80252a9ea476f950ae8f85b0f8f', '17ddf5dd5d51696bb3d7c6291687be6f',
 'e7b71a9017aa05c9a7fd292d714858e8', '5e28dfe12db7fb50a4b2f691faecea5e',
 '56b18e2166679b8a959d72dd06da27f9', '274fa6071e5e17fe303b9748641082c8']
Length: 99441, dtype: str
Values with extra spaces: 0

Column: customer_unique_id
Unique values: <StringArray>
['861eff4711a542e4b93843c6dd7febb0', '290c77bc529b7ac935b93aa66c333dc3',
 '060e732b5

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\58788215.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


Unique values: <StringArray>
[               'franca', 'sao bernardo do campo',             'sao paulo',
       'mogi das cruzes',              'campinas',        'jaragua do sul',
               'timoteo',              'curitiba',        'belo horizonte',
         'montes claros',
 ...
              'balbinos',          'serra bonita',          'venda branca',
           'sanga puita',               'queiroz',                'siriji',
   'natividade da serra',          'monte bonito',            'sao rafael',
     'eugenio de castro']
Length: 4119, dtype: str
Values with extra spaces: 0

Column: customer_state
Unique values: <StringArray>
['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE',
 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE',
 'RR']
Length: 27, dtype: str
Values with extra spaces: 0

DATASET: geolocation

Column: geolocation_city
Unique values: <StringArray>
[            'sao paulo',             'são paulo', 'sao ber

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\58788215.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


Values with extra spaces: 1

Column: geolocation_state
Unique values: <StringArray>
['SP', 'RN', 'AC', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'AL', 'PB', 'CE', 'PI',
 'MA', 'PA', 'AP', 'AM', 'RR', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'RS', 'PR',
 'SC']
Length: 27, dtype: str
Values with extra spaces: 0

DATASET: order_items

Column: order_id
Unique values: <StringArray>
['00010242fe8c5a6d1ba2dd792cb16214', '00018f77f2f0320c557190d7a144bdd3',
 '000229ec398224ef6ca0657da4fc703e', '00024acbcdf0a6daa1e931b038114c75',
 '00042b26cf59d7ce69dfabb4e55b4fd9', '00048cc3ae777c65dbb7d2a0634bc1ea',
 '00054e8431b9d7675808bcb819fb4a32', '000576fe39319847cbb9d288c5617fa6',
 '0005a1a1728c9d785b8e2b08b904576c', '0005f50442cb953dcd1d21e1fb923495',
 ...
 'fffa82886406ccf10c7b4e35c4ff2788', 'fffb0b1a50e65c449020434fa835e078',
 'fffb2ef8874127f75b52b643880fd7e0', 'fffb9224b6fc7c43ebb0904318b10b5f',
 'fffbee3b5462987e66fb49b1c5411df2', 'fffc94f6ce00a00581880bf54a75a037',
 'fffcd46ef2263f404302a634eb57f7eb', 'fffc

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\58788215.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


... (2 outputs left out)

Unique values: <StringArray>
['7bc2406110b926393aa56f80a40eba40', '80e641a11e56f04c1ad469d5645fdfde',
 '228ce5500dc1d8e020d8d1322874b6f0', 'e64fb393e7b32834bb789ff8bb30750e',
 'f7c4243c7fe1938f181bec41a392bdeb', '15197aa66ff4d0650b5434f1b46cda19',
 '07f9bee5d1b850860defd761afa7ff16', '7c6400515c67679fbee952a7525281ef',
 'a3f6f7f6f433de0aefbb97da197c554c', '8670d52e15e00043ae7de4c01cc2fe06',
 ...
 '3c569245c7610d9a51eff50032194ea3', 'a709d176f59bc3af77f4149c96bae357',
 '115d72425fc59eec9a2e9d7594c6d93d', 'c6b270c61f67c9f7cb07d84ea8aeaf8b',
 'af2dc0519de6e0720ef0c74292fb4114', '574ed12dd733e5fa530cfd4bbf39d7c9',
 'f3897127253a9592a73be9bdfdf4ed7a', 'b3de70c89b1510c4cd3d0649fd302472',
 '1adeb9d84d72fe4e337617733eb85149', 'efe49f1d6f951dd88b51e6ccd4cc548f']
Length: 98410, dtype: str
Values with extra spaces: 0

Column: order_id
Unique values: <StringArray>
['73fc7af87114b39712e6da79b0a377eb', 'a548910a1c6147796b98fdf73dbeba33',
 'f9e4b658b201a9f2ecdecbb34bed034b', '658677c97b385a9be1707378

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\58788215.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns



Column: order_purchase_timestamp
Unique values: <StringArray>
['2017-10-02 10:56:33', '2018-07-24 20:41:37', '2018-08-08 08:38:49',
 '2017-11-18 19:28:06', '2018-02-13 21:18:39', '2017-07-09 21:57:05',
 '2017-04-11 12:22:08', '2017-05-16 13:10:30', '2017-01-23 18:29:09',
 '2017-07-29 11:55:02',
 ...
 '2017-10-27 15:21:00', '2017-12-20 09:52:41', '2017-10-04 19:57:37',
 '2017-01-27 00:30:03', '2017-02-23 09:05:12', '2017-03-09 09:54:05',
 '2018-02-06 12:58:58', '2017-08-27 14:46:43', '2018-01-08 21:28:27',
 '2018-03-08 20:57:30']
Length: 98875, dtype: str
Values with extra spaces: 0

Column: order_approved_at
Unique values: <StringArray>
['2017-10-02 11:07:15', '2018-07-26 03:24:27', '2018-08-08 08:55:23',
 '2017-11-18 19:45:59', '2018-02-13 22:20:29', '2017-07-09 22:10:13',
 '2017-04-13 13:25:17', '2017-05-16 13:22:11', '2017-01-25 02:50:47',
 '2017-07-29 12:05:32',
 ...
 '2018-03-04 23:09:22', '2017-10-27 15:32:49', '2017-10-04 20:07:14',
 '2017-01-27 01:05:25', '2017-02-23 09:15:11'

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\58788215.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns
C:\Users\manik\AppData\Local\Temp\ipykernel_14116\58788215.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_

## 9. Most common categorical values

This prints the ten most frequent values in each text column. It is a quick way to understand common categories.

In [9]:
for name, df in data.items():
    print("\nDATASET:", name)

    for col in df.select_dtypes(include="object").columns:
        print("\nColumn:", col)
        print(df[col].value_counts(dropna=False).head(10))



DATASET: customers

Column: customer_id


C:\Users\manik\AppData\Local\Temp\ipykernel_14116\3990897089.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\manik\AppData\Local\Temp\ipykernel_14116\3990897089.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

customer_id
06b8999e2fba1a1fbc88172c00ba8bc7    1
18955e83d337fd6b2def6b18a428ac77    1
4e7b3e00288586ebd08712fdd0374a03    1
b2b6027bc5c5109e529d4dc6358b12c3    1
4f2d8ab171c80ec8364f7c12e35b23ad    1
879864dab9bc3047522c92c82e1212b8    1
fd826e7cf63160e536e0908c76c3f441    1
5e274e7a0c3809e14aba7ad5aae0d407    1
5adf08e34b2e993982a47070956c5c65    1
4b7139f34592b3a31687243a302fa75b    1
Name: count, dtype: int64

Column: customer_unique_id
customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
de34b16117594161a6a89c50b289d35a     6
Name: count, dtype: int64

Column: customer_city
customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte      

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\3990897089.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\manik\AppData\Local\Temp\ipykernel_14116\3990897089.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

order_id
e481f51cbdc54678b7cc49136f2d6af7    1
53cdb2fc8bc7dce0b6741e2150273451    1
47770eb9100c2d0c44946d9cf07ec65d    1
949d5b44dbf5de918fe9c16f97b45f8a    1
ad21c59c0840e6cb83a9ceb5573f8159    1
a4591c265e18cb1dcee52889e2d8acc3    1
136cce7faa42fdb2cefd53fdc79a6098    1
6514b8ad8028c9f2cc2374ded245783f    1
76c6e866289321a7c93b82b54852dc33    1
e69bfb5eb88e0ed6a785585b27e16dbf    1
Name: count, dtype: int64

Column: customer_id
customer_id
9ef432eb6251297304e76186b10a928d    1
b0830fb4747a6c6d20dea0b8c802d7ef    1
41ce2a54c0b03bf3443c3d931a367089    1
f88197465ea7920adcdbec7375364d82    1
8ab97904e6daea8866dbdbc4fb7aad2c    1
503740e9ca751ccdda7ba28e9ab8f608    1
ed0271e0b7da060a393796590e7b737a    1
9bdf08b4b3b52b5526ff42d37d47f222    1
f54a9f0e6b351c431402b8461ea51999    1
31ad1d1b63eb9962463f764d4e6e0c9d    1
Name: count, dtype: int64

Column: order_status
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
proces

C:\Users\manik\AppData\Local\Temp\ipykernel_14116\3990897089.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\manik\AppData\Local\Temp\ipykernel_14116\3990897089.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

## 10. Check duplicate IDs and keys

Some IDs repeat legitimately in detail tables. For example, one order can contain multiple products, so the order-items key uses both `order_id` and `order_item_id`.

In [10]:
key_columns = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category": ["product_category_name"]
}

for name, columns in key_columns.items():
    df = data[name]
    print(name, "duplicate key rows:", df.duplicated(subset=columns).sum())

print("Note: geolocation can legitimately contain repeated ZIP-code prefixes.")

customers duplicate key rows: 0
orders duplicate key rows: 0
order_items duplicate key rows: 0
order_payments duplicate key rows: 0
order_reviews duplicate key rows: 814
products duplicate key rows: 0
sellers duplicate key rows: 0
product_category duplicate key rows: 0
Note: geolocation can legitimately contain repeated ZIP-code prefixes.


## 11. Check whether date values can be parsed

This counts non-empty values that pandas could not convert to dates.

In [11]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for name, columns in date_columns.items():
    df = data[name]

    for col in columns:
        parsed_dates = pd.to_datetime(df[col], errors="coerce")
        bad_dates = df[col].notna().sum() - parsed_dates.notna().sum()
        print(f"{name}.{col}: unparseable dates = {bad_dates}")

orders.order_purchase_timestamp: unparseable dates = 0
orders.order_approved_at: unparseable dates = 0
orders.order_delivered_carrier_date: unparseable dates = 0
orders.order_delivered_customer_date: unparseable dates = 0
orders.order_estimated_delivery_date: unparseable dates = 0
order_reviews.review_creation_date: unparseable dates = 0
order_reviews.review_answer_timestamp: unparseable dates = 0


## 12. Check order date sequence

An order's purchase time should normally be before approval, carrier handoff, and customer delivery. This check uses a copy so the original `orders_df` is not changed.

In [12]:
orders_dates = orders_df.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for col in date_columns:
    orders_dates[col] = pd.to_datetime(orders_dates[col], errors="coerce")

bad_purchase_approval = (
    orders_dates["order_approved_at"] < orders_dates["order_purchase_timestamp"]
)
bad_approval_carrier = (
    orders_dates["order_delivered_carrier_date"] < orders_dates["order_approved_at"]
)
bad_carrier_delivery = (
    orders_dates["order_delivered_customer_date"] < orders_dates["order_delivered_carrier_date"]
)

print("Approval before purchase:", bad_purchase_approval.sum())
print("Carrier handoff before approval:", bad_approval_carrier.sum())
print("Customer delivery before carrier handoff:", bad_carrier_delivery.sum())

Approval before purchase: 0
Carrier handoff before approval: 1359
Customer delivery before carrier handoff: 23


## 13. Check relationships between datasets

These checks find IDs in one table that do not appear in the related table.

In [13]:
relationship_checks = [
    ("orders.customer_id", orders_df, "customer_id", customer_df, "customer_id"),
    ("order_items.order_id", order_items_df, "order_id", orders_df, "order_id"),
    ("order_items.product_id", order_items_df, "product_id", products_df, "product_id"),
    ("order_items.seller_id", order_items_df, "seller_id", sellers_df, "seller_id"),
    ("order_payments.order_id", order_payments_df, "order_id", orders_df, "order_id"),
    ("order_reviews.order_id", order_reviews_df, "order_id", orders_df, "order_id")
]

for label, child_df, child_col, parent_df, parent_col in relationship_checks:
    unmatched = (~child_df[child_col].isin(parent_df[parent_col])).sum()
    print(f"{label}: unmatched IDs = {unmatched}")

orders.customer_id: unmatched IDs = 0
order_items.order_id: unmatched IDs = 0
order_items.product_id: unmatched IDs = 0
order_items.seller_id: unmatched IDs = 0
order_payments.order_id: unmatched IDs = 0
order_reviews.order_id: unmatched IDs = 0


## 14. Check unusual numeric values

These simple rules flag values to investigate. A flagged value is not automatically an error.

In [14]:
value_checks = [
    ("order_items", "price", "negative"),
    ("order_items", "freight_value", "negative"),
    ("order_payments", "payment_value", "negative"),
    ("order_reviews", "review_score", "score"),
    ("products", "product_weight_g", "zero_or_negative"),
    ("products", "product_length_cm", "zero_or_negative"),
    ("products", "product_height_cm", "zero_or_negative"),
    ("products", "product_width_cm", "zero_or_negative")
]

for name, col, kind in value_checks:
    df = data[name]

    if kind == "negative":
        count = (df[col] < 0).sum()
    elif kind == "score":
        count = (~df[col].between(1, 5) & df[col].notna()).sum()
    else:
        count = (df[col] <= 0).sum()

    print(f"{name}.{col}: flagged values = {count}")

order_items.price: flagged values = 0
order_items.freight_value: flagged values = 0
order_payments.payment_value: flagged values = 0
order_reviews.review_score: flagged values = 0
products.product_weight_g: flagged values = 4
products.product_length_cm: flagged values = 0
products.product_height_cm: flagged values = 0
products.product_width_cm: flagged values = 0


## 15. Final checklist

- Missing values are not always errors; some orders may have no delivery date.
- Duplicate rows or IDs should be investigated before removing anything.
- Extra spaces and capitalization differences can create inconsistent categories.
- Very frequent categories help you understand what is common in the dataset.
- Unparseable or out-of-sequence dates should be investigated.
- Unmatched IDs can indicate missing records or relationship problems.
- Zero or negative values should be checked against the meaning of each column.
- Numeric summaries and category counts describe the data; they do not automatically clean it.

**Next step:** After reviewing the flags, decide which issues are genuine errors and handle cleaning in a separate step. This keeps the original data safe.